In [0]:
%pip install gspread google-auth
dbutils.library.restartPython()

In [0]:
# Date range (Time is fixed to 00:00:00 to 23:59:59)
dbutils.widgets.text("from_date", "20/07/2026", "From Date (dd/MM/yyyy)")
dbutils.widgets.text("to_date", "20/07/2026", "To Date (dd/MM/yyyy)")

from_date = dbutils.widgets.get("from_date")
to_date   = dbutils.widgets.get("to_date")

print(f"Daily Backfill range: {from_date} 00:00 -> {to_date} 23:59 (UK local time)")

In [0]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

uk = ZoneInfo("Europe/London")

start_local = datetime.strptime(f"{from_date} 00:00:00", "%d/%m/%Y %H:%M:%S").replace(tzinfo=uk)
end_local   = datetime.strptime(f"{to_date} 23:59:59", "%d/%m/%Y %H:%M:%S").replace(tzinfo=uk)

if end_local < start_local:
    raise ValueError("To date must be after or equal to From date")

# Build each 24-hour block as (start, end) pairs in UK local, plus UTC equivalents for filtering
blocks = []
cursor = start_local
while cursor <= end_local:
    block_start_local = cursor
    # The block ends at 23:59:59.999999 for the current day, OR the user's end time
    block_end_local = min(cursor.replace(hour=23, minute=59, second=59, microsecond=999999), end_local)
    
    # For SQL filtering, it's safer to use < next day 00:00:00
    sql_end_local = block_end_local + timedelta(seconds=1)
    
    blocks.append({
        "start_local": block_start_local,
        "end_local":   block_end_local,
        "start_utc":   block_start_local.astimezone(ZoneInfo("UTC")).strftime("%Y-%m-%d %H:%M:%S"),
        "end_utc":     sql_end_local.astimezone(ZoneInfo("UTC")).strftime("%Y-%m-%d %H:%M:%S"),
    })
    # Move cursor to the start of the next day
    cursor = (cursor + timedelta(days=1)).replace(hour=0, minute=0, second=0, microsecond=0)

print(f"{len(blocks)} daily blocks to backfill:")
for b in blocks:
    print(f"  {b['start_local'].strftime('%d/%m/%Y')} 00:00 - {b['end_local'].strftime('%d/%m/%Y')} 23:59")

In [0]:
import gspread
from google.oauth2.service_account import Credentials
import json

KEY_FILE_PATH = "/Workspace/Users/pakhei_tsang@next.co.uk/advance-mantis-398714-2168c9162641.json"  # adjust to your path

with open(KEY_FILE_PATH, "r") as f:
    creds_dict = json.load(f)

creds = Credentials.from_service_account_info(
    creds_dict, scopes=["https://www.googleapis.com/auth/spreadsheets"]
)
gc = gspread.authorize(creds)

SHEET_ID = "1OOvBCBOB2iya_voLFwgr3kG0X9f9Aqqo2gogdStzFac"
sh = gc.open_by_key(SHEET_ID)
ws = sh.worksheet("MPF - VS Data")

print("Connected to:", sh.title, "-> Tab:", ws.title)

In [0]:
# Read directly from the ABFSS delta path
df = (spark.read
      .format("delta")
      .load("abfss://landing@whsanalyticsdlsprodeuw.dfs.core.windows.net/streaming/landing_bonushub_event_parsed/delta/"))
df.createOrReplaceTempView("landing_bonus_hub_event_parsed")

all_reports = [
    # Updated Area to 'Picking' and using EXISTS for the array
    {"name": "MPF - VS Transfer", "area": "('Picking')", "event": None, "extra": "EXISTS(PAYLOAD_ATTRIBUTES, x -> x LIKE '%Stream_B%')"},
    {"name": "MPF - VS Retail",   "area": "('Picking')", "event": None, "extra": "EXISTS(PAYLOAD_ATTRIBUTES, x -> x LIKE '%Stream_R%')"},
    {"name": "MPF - VS Online",   "area": "('Picking')", "event": None, "extra": "EXISTS(PAYLOAD_ATTRIBUTES, x -> x LIKE '%Stream_O%')"}, 
]

# Exact columns matching your requirement (including the 3 empty string placeholders)
COLUMNS = [
    'PAYLOAD_EVENTTYPE', 'Total_Quantity', 'Total_StandardHours', 'Total_SMV', 
    '', '', '', 
    'Week', 'Date Time Range', 'Report Name'
]

print(f"Reports to run: {len(all_reports)}")
for r in all_reports:
    print(f"  - {r['name']}")

In [0]:
import pandas as pd

failures = []
total_rows_pushed = 0

for b in blocks:
    # Per-block metadata, matching the live job's format exactly
    week_val = spark.sql(f"""
      SELECT MOD(FLOOR(DATEDIFF(to_date(from_utc_timestamp(timestamp('{b['start_utc']}'),'Europe/London')), DATE'2025-12-14')/7)+46,52) AS w
    """).collect()[0]['w']

    date_time_range = f"{b['start_local'].strftime('%d/%m/%Y %H:%M')} - {b['end_local'].strftime('%d/%m/%Y %H:%M')}"
    date_display = b['start_local'].strftime('%d/%m/%Y')

    print(f"\n=== Block: {date_time_range} ===")

    for r in all_reports:
        try:
            area_clause = f"AND PAYLOAD_AREACODE IN {r['area']}"
            event_clause = f"AND PAYLOAD_EVENTTYPE IN {r['event']}" if r['event'] else ""
            extra_clause = f"AND {r['extra']}" if r['extra'] else ""

            query = f"""
              SELECT
                PAYLOAD_EVENTTYPE,
                SUM(PAYLOAD_QUANTITY)      AS Total_Quantity,
                SUM(PAYLOAD_STANDARDHOURS) AS Total_StandardHours,
                SUM(PAYLOAD_SMV)           AS Total_SMV
              FROM landing_bonus_hub_event_parsed
              WHERE TRIM(PAYLOAD_WAREHOUSECODE) = 'R'
                {area_clause}
                {event_clause}
                {extra_clause}
                AND to_timestamp(PAYLOAD_EVENTTIMESTAMP) >= timestamp('{b['start_utc']}')
                AND to_timestamp(PAYLOAD_EVENTTIMESTAMP) <  timestamp('{b['end_utc']}')
              GROUP BY 1
              ORDER BY PAYLOAD_EVENTTYPE
            """

            df_r = spark.sql(query).toPandas()
            
            # Assign the metadata and empty placeholder columns to match the COLUMNS list exactly
            df_r[''] = ''
            df_r['Week'] = week_val
            df_r['Date Time Range'] = date_time_range
            df_r['Report Name'] = r['name']
            
            # Reorder to match COLUMNS exactly
            df_r = df_r[COLUMNS]

            if len(df_r) == 0:
                df_r = pd.DataFrame(
                    [[ '(no data this window)', '', '', '', '', '', '', week_val, date_time_range, r['name'] ]],
                    columns=COLUMNS
                )

            values = df_r.astype(str).values.tolist()
            ws.append_rows(values, value_input_option='USER_ENTERED', table_range='A1')
            total_rows_pushed += len(df_r)
            print(f"  [{r['name']}] {len(df_r)} rows")

        except Exception as e:
            error_msg = f"{type(e).__name__}: {str(e)}"
            print(f"  [{r['name']}] FAILED — {error_msg}")
            failures.append({"block": date_time_range, "report": r['name'], "error": error_msg})
            continue

print(f"\n--- Backfill summary ---")
print(f"Blocks processed: {len(blocks)}")
print(f"Total rows pushed: {total_rows_pushed}")
print(f"Failures: {len(failures)}")
for f in failures:
    print(f"  {f['block']} | {f['report']} | {f['error']}")